# Agents Module Notebook

This notebook provides a guided walkthrough of the core agent patterns used in `langChain/agents/`. It focuses on:

1. Basic LangChain agents with tools
2. LangGraph-based orchestration
3. Langfuse tracing for observability
4. Advanced traced graph workflows

The A2A module is intentionally not covered as a runnable notebook workflow because it depends on multiple long-running services in separate terminals.

## Learning Objectives

By the end of this notebook, you should be able to:

- explain the difference between direct model calls and tool-using agents
- build a simple LangChain agent with `create_agent`
- understand when LangGraph is useful for agent workflows
- add tracing with Langfuse
- compare simple agent execution with graph-based execution


## Environment Setup

Before running the notebook:

- Ensure `sandbox.yaml` is configured
- Load any required values in `.env`
- If you want to run the Langfuse sections, set `LANGFUSE_HOST`, `LANGFUSE_PK`, and `LANGFUSE_SK`
- configure cwd for jupyter match your workspace python code: 
    -  vscode menu -> Settings > Extensions > Jupyter > Notebook File Root
    -  change from `${fileDirname}` to `${workspaceFolder}`

The Langfuse sections are optional and can be skipped if you only want to study the core agent patterns.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from envyaml import EnvYAML

load_dotenv()
sys.path.insert(0, str(Path('langChain').resolve()))
from oci_openai_helper import OCIOpenAIHelper

SANDBOX_CONFIG_FILE = 'sandbox.yaml'
scfg = EnvYAML(SANDBOX_CONFIG_FILE)
print('Configuration loaded.')

## Part 1 — Basic LangChain Agent

We start with the simplest agent pattern in this module: a LangChain agent created with `create_agent` and a small set of tools.

This mirrors the ideas in `langchain_agent.py`, but in a teaching-oriented notebook format.

In [ ]:
from typing import Any
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_core.tools import tool

LLM_MODEL = 'xai.grok-4-fast-non-reasoning'
llm = OCIOpenAIHelper.get_langchain_openai_client(model_name=LLM_MODEL, config=scfg)

@tool
def get_weather(zipcode: int, date: str) -> dict[str, Any]:
    """Return sample weather data for a zipcode and date."""
    return {'rain': True, 'min_temperature': '50 F', 'max_temperature': '62 F'}

@tool
def get_city(criteria: str) -> dict[str, Any]:
    """Recommend a city based on the supplied criteria."""
    return {'city_name': 'Chicago', 'zipcode': 60601}

@tool
def get_clothes(gender: str, temp: int, rain: bool) -> dict[str, list[str]]:
    """Suggest clothing and accessories."""
    return {'clothes': ['rain coat', 'jeans', 'formal shirt'], 'accessories': ['umbrella', 'boots']}

tools = [get_weather, get_city, get_clothes]
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt='Answer questions by using the provided tools. Some queries may require multiple tool calls. Do not use your general knowledge, only use provided tools aggressively for each step'
)
print('Basic agent ready.')

In [ ]:
MESSAGE = 'What types of clothes should I wear on a trip to Oracle headquarters next week?'
result = agent.invoke({'messages': [HumanMessage(MESSAGE)]})
result['messages'][-1].content

### Try It

Experiments to try in this section:

- Change the prompt and see whether the tool sequence changes
- Add another tool
- Change the hardcoded tool outputs
- Switch the model to a different supported option


## Part 2 — LangGraph Agent

Now we move from a simple LangChain agent to a graph-based workflow. This gives you more control over how tool execution and routing are handled.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langchain.messages import SystemMessage, ToolMessage

llm_with_tools = llm.bind_tools(tools)
tools_by_name = {tool.name: tool for tool in tools}

def llm_call(state: MessagesState):
    return {
        'messages': [
            llm_with_tools.invoke(
                [SystemMessage(content='You are a helpful assistant. Infer missing details when reasonable.')]
                + state['messages']
            )
        ]
    }

def tool_node(state: MessagesState):
    result = []
    tool_calls = getattr(state['messages'][-1], 'tool_calls', [])
    for tool_call in tool_calls:
        selected_tool = tools_by_name[tool_call['name']]
        observation = selected_tool.invoke(tool_call['args'])
        result.append(ToolMessage(content=str(observation), tool_call_id=tool_call['id']))
    return {'messages': result}

def should_continue(state: MessagesState):
    last_message = state['messages'][-1]
    if getattr(last_message, 'tool_calls', []):
        return 'tool_node'
    return END

builder = StateGraph(MessagesState)
builder.add_node('llm_call', llm_call)
builder.add_node('tool_node', tool_node)
builder.add_edge(START, 'llm_call')
builder.add_conditional_edges('llm_call', should_continue, ['tool_node', END])
builder.add_edge('tool_node', 'llm_call')
graph_agent = builder.compile(checkpointer=InMemorySaver())
print('LangGraph agent ready.')

In [ ]:
graph_result = graph_agent.invoke(
    {'messages': [HumanMessage(MESSAGE)]},
    config={'configurable': {'thread_id': '1'}}
)
graph_result['messages'][-1].content

### Why LangGraph?

LangGraph is useful when you want:

- explicit control over routing
- multi-step workflows
- custom nodes or summary steps
- easier extension into more complex orchestration


## Part 3 — Langfuse Tracing

Tracing helps you observe how an agent behaves: what tools it called, what metadata was attached, and how the final response was produced.

If you do not have Langfuse configured, you can skip this section.

In [ ]:
langfuse_enabled = all(scfg.get('langfuse', {}).get(k) for k in ['langfuse_pk', 'langfuse_sk', 'langfuse_host'])
langfuse_enabled

In [ ]:
if langfuse_enabled:
    import datetime
    from langfuse import Langfuse, observe
    from langfuse.langchain import CallbackHandler

    langfuse = Langfuse(
        public_key=scfg['langfuse']['langfuse_pk'],
        secret_key=scfg['langfuse']['langfuse_sk'],
        host=scfg['langfuse']['langfuse_host'],
    )
    langfuse_handler = CallbackHandler()
    traced_config = {
        'callbacks': [langfuse_handler],
        'metadata': {
            'langfuse_user_id': os.getenv('MY_PREFIX', 'default_user'),
            'langfuse_session_id': datetime.datetime.now().strftime('%Y-%m-%d_%H-%M'),
            'langfuse_tags': ['workshop', os.getenv('MY_PREFIX', 'user-name')],
        },
    }
    traced_result = agent.invoke({'messages': [HumanMessage(MESSAGE)]}, config=traced_config)
    display(traced_result['messages'][-1].content)
else:
    print('Langfuse is not configured. Skip this section.')

## Part 4 — Advanced Traced Graph Pattern

The advanced traced graph pattern builds on LangGraph and adds richer observability. This is the conceptual pattern used in `langfuse_graph.py`.

In practice, this pattern is useful when you want:
- trace metadata per node
- custom annotations with `@observe()`
- multi-step or multi-agent graphs


### Conceptual Next Step

A traced graph typically adds:

- `@observe()` on key graph nodes
- manual calls like `langfuse.update_current_trace(...)`
- a second client or summary node
- richer per-run metadata for debugging

See `langfuse_graph.py` for the full script example.

## Comparison Summary

| Pattern | Best for | Tradeoff |
|---|---|---|
| `create_agent` | Simple tool-using assistants | Less workflow control |
| LangGraph | Multi-step orchestration and routing | More code and structure |
| Langfuse tracing | Debugging and observability | Requires extra setup |
| Traced graph workflows | Advanced orchestration visibility | Most complexity |


## Exercises

Try one or more of these:

1. Add a new tool and bind it to both the simple agent and the graph agent
2. Change the prompt and compare tool usage patterns
3. Swap the model for a reasoning model and compare results
4. If Langfuse is enabled, add your own tags and user metadata
5. Extend the graph with an additional node


## Where to Go Next

After this notebook:

- review the script versions in `langChain/agents/`
- explore the A2A module in `langChain/agents/a2a/`
- note that the A2A examples are intentionally script-driven because they require multiple cooperating services to run in parallel
